---
## Section 0: Imports

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats


sys.path.append("../../utils")
from dataset import get_project_root, load_csv

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid")

current = Path.cwd().resolve()

while current != current.parent:
    if (current / ".git").exists() or (current / "pyproject.toml").exists():
        project_root = current
        break
    current = current.parent

PROJ_ROOT = get_project_root()
OUTPUT_PATH = PROJ_ROOT / "data" / "support2_cleaned.csv"
print(f"Output path  : {OUTPUT_PATH}")

Output path  : C:\Users\mwee\Documents\USD-AAI\aai-500\final\usd-aai-500-final\data\support2_cleaned.csv


---
## Section 1: Load Data & Initial Inspection

In [ ]:
df_raw = load_csv("support2_raw_complete.csv")
print(f"Shape: {df_raw.shape[0]:,} rows  x  {df_raw.shape[1]} columns")
df_raw.head()

Shape: 9,105 rows  x  48 columns


,id,age,death,sex,hospdead,slos,d.time,dzgroup,dzclass,num.co,edu,income,scoma,charges,totcst,totmcst,avtisst,race,sps,aps,surv2m,surv6m,hday,diabetes,dementia,ca,prg2m,prg6m,dnr,dnrday,meanbp,wblc,hrt,resp,temp,pafi,alb,bili,crea,sod,ph,glucose,bun,urine,adlp,adls,sfdm2,adlsc
0,1,62.8500,0,male,0,5,2029,Lung Cancer,Cancer,0,11.0000,$11-$25k,0.0000,9715.0000,NaN,NaN,7.0000,other,33.8984,20.0000,0.2629,0.0370,1,0,0,metastatic,0.5000,0.2500,no dnr,5.0000,97.0000,6.0000,69.0000,22.0000,36.0000,388.0000,1.7998,0.2000,1.2000,141.0000,7.4600,NaN,NaN,NaN,7.0000,7.0000,NaN,7.0000
1,2,60.3390,1,female,1,4,4,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0000,$11-$25k,44.0000,34496.0000,NaN,NaN,29.0000,white,52.6953,74.0000,0.0010,0.0000,3,0,0,no,0.0000,0.0000,NaN,NaN,43.0000,17.0977,112.0000,34.0000,34.5938,98.0000,NaN,NaN,5.5000,132.0000,7.2500,NaN,NaN,NaN,NaN,1.0000,<2 mo. follow-up,1.0000
2,3,52.7470,1,female,0,17,47,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0000,under $11k,0.0000,41094.0000,NaN,NaN,13.0000,white,20.5000,45.0000,0.7909,0.6649,4,0,0,no,0.7500,0.5000,no dnr,17.0000,70.0000,8.5000,88.0000,28.0000,37.3984,231.6562,NaN,2.1997,2.0000,134.0000,7.4600,NaN,NaN,NaN,1.0000,0.0000,<2 mo. follow-up,0.0000
3,4,42.3850,1,female,0,3,133,Lung Cancer,Cancer,2,11.0000,under $11k,0.0000,3075.0000,NaN,NaN,7.0000,white,20.0977,19.0000,0.6990,0.4120,1,0,0,metastatic,0.9000,0.5000,no dnr,3.0000,75.0000,9.0996,88.0000,32.0000,35.0000,NaN,NaN,NaN,0.7999,139.0000,NaN,NaN,NaN,NaN,0.0000,0.0000,no(M2 and SIP pres),0.0000
4,5,79.8850,0,female,0,16,2029,ARF/MOSF w/Sepsis,ARF/MOSF,1,NaN,NaN,26.0000,50127.0000,NaN,NaN,18.6667,white,23.5000,30.0000,0.6349,0.5330,3,0,0,no,0.9000,0.9000,no dnr,16.0000,59.0000,13.5000,112.0000,20.0000,37.8984,173.3125,NaN,NaN,0.7999,143.0000,7.5098,NaN,NaN,NaN,NaN,2.0000,no(M2 and SIP pres),2.0000


In [6]:
df_raw.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 9105 entries, 0 to 9104
Data columns (total 48 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        9105 non-null   int64  
 1   age       9105 non-null   float64
 2   death     9105 non-null   int64  
 3   sex       9105 non-null   str    
 4   hospdead  9105 non-null   int64  
 5   slos      9105 non-null   int64  
 6   d.time    9105 non-null   int64  
 7   dzgroup   9105 non-null   str    
 8   dzclass   9105 non-null   str    
 9   num.co    9105 non-null   int64  
 10  edu       7471 non-null   float64
 11  income    6123 non-null   str    
 12  scoma     9104 non-null   float64
 13  charges   8933 non-null   float64
 14  totcst    8217 non-null   float64
 15  totmcst   5630 non-null   float64
 16  avtisst   9023 non-null   float64
 17  race      9063 non-null   str    
 18  sps       9104 non-null   float64
 19  aps       9104 non-null   float64
 20  surv2m    9104 non-null   float64
 21  su

In [ ]:
int_cols = df_raw.select_dtypes(include="int64").columns
binary_cols = [
    c
    for c in int_cols
    if df_raw[c].nunique() == 2 and set(df_raw[c].unique()) == {0, 1}
]
print("===========Binary Columns==========")
print(binary_cols)


===========Binary Columns==========
['death', 'hospdead', 'diabetes', 'dementia']


### 1.1 Descriptive Statistics

In [10]:
desc = df_raw.describe().T
desc.columns = [
    "Count",
    "Mean",
    "Std Dev",
    "Min",
    "Q1 (25%)",
    "Median (50%)",
    "Q3 (75%)",
    "Max",
]
desc

,Count,Mean,Std Dev,Min,Q1 (25%),Median (50%),Q3 (75%),Max
id,9105.0000,4553.0000,2628.5314,1.0000,2277.0000,4553.0000,6829.0000,9105.0000
age,9105.0000,62.6508,15.5937,18.0420,52.7970,64.8570,73.9990,101.8480
death,9105.0000,0.6811,0.4661,0.0000,0.0000,1.0000,1.0000,1.0000
hospdead,9105.0000,0.2592,0.4382,0.0000,0.0000,0.0000,1.0000,1.0000
slos,9105.0000,17.8630,22.0064,3.0000,6.0000,11.0000,20.0000,343.0000
d.time,9105.0000,478.4499,560.3833,3.0000,26.0000,233.0000,761.0000,2029.0000
num.co,9105.0000,1.8686,1.3444,0.0000,1.0000,2.0000,3.0000,9.0000
edu,7471.0000,11.7477,3.4477,0.0000,10.0000,12.0000,14.0000,31.0000
scoma,9104.0000,12.0585,24.6367,0.0000,0.0000,0.0000,9.0000,100.0000
charges,8933.0000,59995.7878,102648.7782,1169.0000,9740.0000,25024.0000,64598.0000,1435423.0000


---
## Section 2: Column Renaming

Rename some columsn to follow PEP-8 standard

In [7]:
rename_map = {
    "d.time": "d_time",
    "num.co": "num_co",
}
df = df_raw.rename(columns=rename_map).copy()

print("Column renames applied:")
for old, new in rename_map.items():
    print(f"  {old!r}  ->  {new!r}")
print(f"\nTotal columns: {len(df.columns)}")

Column renames applied:
  'd.time'  ->  'd_time'
  'num.co'  ->  'num_co'

Total columns: 48


---
## Section 3: Target Variable

**Did the patient die within 180 days of study?**

To answer this we look at two fields: 
- `death`:  whether the patient ever died during 180 days (1 = yes, 0 = no)
- `d_time`:  days of follow-up (how long they were observed). If this number is greater than 180 days of the trial, that means they survived the trial

death_180x = { death = 1 && d_time <= 180 }

Notice the AND conditional because the patient must've died within the 180 days to satisfy our target. 
We then use **death_180d** for binary classification


In [11]:
df["death_180d"] = ((df["death"] == 1) & (df["d_time"] <= 180)).astype(int)

counts = df["death_180d"].value_counts().sort_index()
rel_freqs = df["death_180d"].value_counts(normalize=True).sort_index()

summary = pd.DataFrame(
    {
        "Outcome": ["Survived >= 180 days (0)", "Died within 180 days (1)"],
        "Frequency": counts.values,
        "Relative Frequency": rel_freqs.values.round(4),
        "Percentage (%)": (rel_freqs.values * 100).round(1),
    }
)
print(summary.to_string(index=False))
print(f"\nClass imbalance ratio (survived: died) = {counts[0] / counts[1]:.2f} : 1")

                 Outcome  Frequency  Relative Frequency  Percentage (%)
Survived >= 180 days (0)       4840              0.5316         53.2000
Died within 180 days (1)       4265              0.4684         46.8000

Class imbalance ratio (survived: died) = 1.13 : 1


---
## Section 4: Data Type Casting (into Pandas Type)

Correct data types are for proper computing and modeling.
We apply three categories of correction. Casting to Int8 also handles NaN gracefully:

1. **Binary integer columns** -> `Int8` (nullable integer; memory-efficient)
2. **Nominal categorical columns** -> unordered `pd.Categorical`
3. **Ordinal categorical columns** -> ordered `pd.Categorical` so that sort
   operations and comparisons respect the natural logical order

Per documentation:

**`income` ordering (ascending socioeconomic status):**
`under $11k` < `$11-$25k` < `$25-$50k` < `>$50k`

**`sfdm2` ordering (best functional status → worst):**
`no disability` < `moderate ADL impairment` < `severe SIP impairment` < `coma/intubated` < `died before 2-month interview`

In [18]:
# Binary columns -> nullable Int8
binary_cols = ["death", "hospdead", "diabetes", "dementia"]
for col in binary_cols:
    df[col] = df[col].astype("Int8")

######  8 Catagoricals
# sex = 2 levels
# dzgroup = 5
# dzclass = 4
# race = 5
# income = 4
# ca = 3
# dnr = 3
# sfdm2 = 5
##############

# Nominal (unordered) categoricals
nominal_cats = ["sex", "dzgroup", "dzclass", "race", "ca", "dnr"]
for col in nominal_cats:
    df[col] = df[col].astype("category")

# Ordinal: income (lowest -> highest bracket) as coded in support2_data_dictionary.md
income_order = ["under $11k", "$11-$25k", "$25-$50k", ">$50k"]
df["income"] = pd.Categorical(df["income"], categories=income_order, ordered=True)

# Ordinal: sfdm2 (2-month functional disability; no disability -> died before interview)
# as coded in support2_data_dictionary.md
sfdm2_order = [
    "no(M2 and SIP pres)",  # Level 1 - no moderate/severe disability
    "adl>=4 (>=5 if sur)",  # Level 2 - unable to do 4+ ADL
    "SIP>=30",  # Level 3 - Sickness Impact Profile >= 30
    "Coma or Intub",  # Level 4 - intubated or in coma at 2 months
    "<2 mo. follow-up",  # Level 5 - died before 2-month interview
]
df["sfdm2"] = pd.Categorical(df["sfdm2"], categories=sfdm2_order, ordered=True)

print("Data type corrections applied:\n")
cols_updated = binary_cols + nominal_cats + ["income", "sfdm2"]
print(df[cols_updated].dtypes.to_string())

Data type corrections applied:

death           Int8
hospdead        Int8
diabetes        Int8
dementia        Int8
sex         category
dzgroup     category
dzclass     category
race        category
ca          category
dnr         category
income      category
sfdm2       category
